In [32]:
import os

In [33]:
pwd

'/Users/basazinbelhu/Simple_MLOPS'

In [34]:
os.chdir("/Users/basazinbelhu/Simple_MLOPS")

In [36]:
pwd

'/Users/basazinbelhu/Simple_MLOPS'

In [37]:
import pandas as pd
import pandas as pd
data = pd.read_csv("artifact/data_ingestion/winequality-red.csv", sep=";")
data.head()

FileNotFoundError: [Errno 2] No such file or directory: 'artifact/data_ingestion/winequality-red.csv'

In [38]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [39]:
data.isnull().sum()

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

In [40]:
data.shape

(1599, 12)

In [41]:
from dataclasses import dataclass
from pathlib import Path
@dataclass
class DataValidationConfig:
    root_dir: Path
    unzip_data_dir: Path
    STATUS_FILE: str
    all_schema: dict

In [42]:
from src.simple_mlops.constants import *
from src.simple_mlops.utils.common import read_yaml, create_directories

In [44]:
class ConfigurationManager:
    def __init__(
        self,
        config_file_path=CONFIG_FILE_PATH,
        params_file_path=PARAMS_FILE_PATH,
        schema_file_path=SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config.artifact_dir])

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        schema = self.schema.COLUMNS

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir=config.root_dir,
            unzip_data_dir=config.unzip_data_dir,
            STATUS_FILE=config.STATUS_FILE,
            all_schema=schema,
        )

        return data_validation_config

In [45]:
import os
from src.simple_mlops import logger

In [46]:
import pandas as pd
from src.simple_mlops import logger
# from src.simple_mlops.entity.config_entity import DataValidationConfig


class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_all_columns(self) -> bool:
        """
        Validates that the dataset's columns and dtypes match the schema.
        Writes the result to STATUS_FILE and returns the status.
        """
        try:
            validation_status = True

            data = pd.read_csv(self.config.unzip_data_dir, sep=";")  # semicolon-delimited
            all_cols = list(data.columns)
            schema_cols = self.config.all_schema.keys()

            # 1) every column in the data must be declared in the schema
            for col in all_cols:
                if col not in schema_cols:
                    validation_status = False
                    logger.info(f"Unexpected column not in schema: {col}")

            # 2) every column in the schema must be present in the data
            for col in schema_cols:
                if col not in all_cols:
                    validation_status = False
                    logger.info(f"Missing expected column: {col}")

            # 3) dtypes must match the schema
            for col, expected_dtype in self.config.all_schema.items():
                if col in data.columns:
                    actual_dtype = str(data[col].dtype)
                    if actual_dtype != expected_dtype:
                        validation_status = False
                        logger.info(
                            f"Dtype mismatch in '{col}': "
                            f"expected {expected_dtype}, got {actual_dtype}"
                        )

            with open(self.config.STATUS_FILE, "w") as f:
                f.write(f"Validation status: {validation_status}")

            logger.info(f"Validation status: {validation_status}")
            return validation_status

        except Exception as e:
            raise e

In [47]:
try:
    config_manager = ConfigurationManager()
    data_validation_config = config_manager.get_data_validation_config()
    data_validator = DataValidation(config=data_validation_config)
    validation_result = data_validator.validate_all_columns()
    logger.info(f"Data validation completed with result: {validation_result}")
except Exception as e:
    logger.error(f"Data validation failed: {e}")

[2026-06-12 18:42:20,487: INFO: common]: Successfully read YAML file: config/config.yaml
[2026-06-12 18:42:20,488: INFO: common]: Successfully read YAML file: params.yaml
[2026-06-12 18:42:20,491: INFO: common]: Successfully read YAML file: schema.yaml
[2026-06-12 18:42:20,492: INFO: common]: Directory created or already exists: artifact
[2026-06-12 18:42:20,493: INFO: common]: Directory created or already exists: artifact/data_validation
[2026-06-12 18:42:20,494: ERROR: 2213151988]: Data validation failed: [Errno 2] No such file or directory: 'artifact/data_ingestion/winequality-red.csv'
